# Food Long-Tail OOD Challenge — Vanilla ResNet Baseline

Plainest possible baseline:
1. ResNet-50 ImageNet pretrained, fine-tune on the full training set (no long-tail tricks).
2. Predict the argmax over 101 known classes for every test image.
3. Write `submission.csv`.

Expected behavior: the model can never predict the `unknown` class (101), so the ~25% OOD test samples will all be wrong — capping the achievable Top-1 accuracy at roughly 75%. This baseline establishes the floor; better submissions add long-tail and OOD handling on top.

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# 直接拉取你们的比赛数据
!kaggle competitions download -c ow-food

# 将下载好的压缩包解压到专门的文件夹中
!unzip -q ow-food.zip -d /content/Food-Classification

print("✅ 数据集极速下载与解压完毕！")

100% 853M/853M [00:48<00:00, 18.4MB/s]

✅ 数据集极速下载与解压完毕！


## 1. Setup

In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm.auto import tqdm

# 数据根目录与提交输出路径
DATA_ROOT = Path('/content/Food-Classification')
OUT_SUB = DATA_ROOT / 'submission(6).csv'

# 已知类数：模型只学 0..100，OOD（101）由后处理判定
NUM_KNOWN = 101
IMG_SIZE = 224          # ResNet 默认输入；图像本身是 256，会被 resize 到 224
BATCH_SIZE = 64
NUM_WORKERS = 4
EPOCHS = 5              # baseline 跑通即可，不追求收敛
LR = 1e-3
SEED = 42

# 自动选设备：优先 CUDA → Apple MPS → CPU
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')
print('Device:', DEVICE)

torch.manual_seed(SEED); np.random.seed(SEED)

Device: cuda


## 2. Dataset

In [ ]:
IMNET_MEAN = [0.485, 0.456, 0.406]
IMNET_STD = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(IMNET_MEAN, IMNET_STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMNET_MEAN, IMNET_STD),
])

class FoodDataset(Dataset):
    """从 csv (image_id[, label]) + 图像目录加载样本。
    has_label=False 用于测试集，__getitem__ 第二项返回 image_id 以便回填。"""

    def __init__(self, df, img_dir, transform, has_label):
        self.df = df.reset_index(drop=True)
        self.img_dir = Path(img_dir)
        self.transform = transform
        self.has_label = has_label

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(self.img_dir / f"{row['image_id']}.jpg").convert('RGB')
        img = self.transform(img)
        if self.has_label:
            return img, int(row['label'])
        return img, row['image_id']

# 现在程序知道 DATA_ROOT 是什么了，再安全地读取 CSV
train_df = pd.read_csv(DATA_ROOT / 'train.csv')
test_df = pd.read_csv(DATA_ROOT / 'test.csv')
print(f'✅ 表格读取成功！train: {len(train_df)}, test: {len(test_df)}')

# 加载图片数据集
tr_ds = FoodDataset(train_df, DATA_ROOT / 'train_images' / 'train_images', train_tf, has_label=True)
test_ds = FoodDataset(test_df, DATA_ROOT / 'test_images' / 'test_images', eval_tf, has_label=False)

# pin_memory 仅在 CUDA 下能加速，MPS/CPU 无效
tr_loader = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True,
                       num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=='cuda'))
# 推理时 batch 翻倍，无需反向传播显存压力小
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE*2, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=='cuda'))
print('✅ 数据流水线组装完毕，可以开始训练了！')

✅ 表格读取成功！train: 32856, test: 20179
✅ 数据流水线组装完毕，可以开始训练了！


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


## 3. Model

In [ ]:
# ResNet-50 + ImageNet V2 权重（精度比 V1 高约 1%）
# 替换 fc 层，输出维度改为 101（已知类数）
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
model.fc = nn.Linear(model.fc.in_features, NUM_KNOWN)
model = model.to(DEVICE)
print(f'ResNet-50, output dim {NUM_KNOWN}')

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 172MB/s]


ResNet-50, output dim 101


## 4. Train

In [ ]:
import numpy as np
import time
import torch
import torch.nn as nn
from tqdm.auto import tqdm

# =====================================================================
# 🎯 核心大招：计算并注入长尾逆频率权重
# =====================================================================
print("⚖️ 正在计算长尾类别分布权重...")
# 1. 统计每个类别的数量并按索引 0-100 排序
class_counts = train_df['label'].value_counts().sort_index().values

# 🌟 改动 1：使用平方根平滑（Smooth），让 100 倍的差距缩减到 10 倍左右，防止模型崩溃
weights = 1.0 / (np.sqrt(class_counts) + 1.0)
weights = weights / np.sum(weights) * NUM_KNOWN
class_weights = torch.FloatTensor(weights).to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=class_weights)
print("✅ 平衡交叉熵损失 (Balanced Cross Entropy) 加载完毕！")
# =====================================================================

# 🌟 改动 2：将微调学习率从 1e-3 降低到 1e-4（或者 5e-5），温柔地保护预训练权重
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

# 开始正式的训练循环
for epoch in range(1, EPOCHS + 1):
    model.train()
    t0 = time.time()
    total, correct, loss_sum = 0, 0, 0.0
    bar = tqdm(tr_loader, desc=f'epoch {epoch}/{EPOCHS}', leave=False)

    for x, y in bar:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        logits = model(x)
        loss = criterion(logits, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        loss_sum += loss.item() * x.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        total += x.size(0)
        bar.set_postfix(loss=loss_sum/total, acc=correct/total)

    scheduler.step()
    print(f'Epoch {epoch}/{EPOCHS} | loss {loss_sum/total:.3f} | train acc {correct/total:.3f} | {time.time()-t0:.1f}s')

⚖️ 正在计算长尾类别分布权重...
✅ 平衡交叉熵损失 (Balanced Cross Entropy) 加载完毕！


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


epoch 1/5:   0%|          | 0/514 [00:00<?, ?it/s]

Epoch 1/5 | loss 2.710 | train acc 0.489 | 318.8s


epoch 2/5:   0%|          | 0/514 [00:00<?, ?it/s]

...

Epoch 3/5 | loss 0.828 | train acc 0.812 | 321.4s


epoch 4/5:   0%|          | 0/514 [00:00<?, ?it/s]

Epoch 4/5 | loss 0.513 | train acc 0.881 | 322.7s


epoch 5/5:   0%|          | 0/514 [00:00<?, ?it/s]

Epoch 5/5 | loss 0.362 | train acc 0.917 | 322.3s


In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
from tqdm.auto import tqdm

# ============================================================
# 方法一：针对 torchvision ResNet，手动提取 fc 前的特征
# ============================================================

NUM_CLASSES = NUM_KNOWN  # 一般就是 101

def extract_resnet_features(model, x):
    """
    提取 ResNet 最后一层 fc 之前的特征。
    输入:
        x: [batch_size, 3, H, W]
    输出:
        feats: [batch_size, 2048]
    """
    x = model.conv1(x)
    x = model.bn1(x)
    x = model.relu(x)
    x = model.maxpool(x)

    x = model.layer1(x)
    x = model.layer2(x)
    x = model.layer3(x)
    x = model.layer4(x)

    x = model.avgpool(x)        # [batch_size, 2048, 1, 1]
    x = torch.flatten(x, 1)     # [batch_size, 2048]

    return x


# ============================================================
# 提取训练集特征，并计算每个类别的特征中心
# ============================================================

print("正在提取训练集特征，并计算每一类的特征中心...")

model.eval()

features_per_class = [[] for _ in range(NUM_CLASSES)]

with torch.no_grad():
    for x, y in tqdm(tr_loader, desc="Extract train features"):
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        feats = extract_resnet_features(model, x)  # [batch_size, 2048]

        # 可选：先归一化特征，后面用 cosine similarity 会更稳定
        feats = F.normalize(feats, dim=1)

        for i in range(x.size(0)):
            label = y[i].item()
            features_per_class[label].append(feats[i].cpu().numpy())


# 计算每一类的中心向量
class_centers = []

for c in range(NUM_CLASSES):
    if len(features_per_class[c]) == 0:
        print(f"警告：类别 {c} 没有训练样本，使用零向量代替")
        center = np.zeros(2048, dtype=np.float32)
    else:
        feats_c = np.stack(features_per_class[c], axis=0)
        center = feats_c.mean(axis=0)

    class_centers.append(center)

class_centers = np.stack(class_centers, axis=0)  # [101, 2048]
class_centers = torch.tensor(class_centers, dtype=torch.float32).to(DEVICE)

# 对类别中心也做一次归一化，方便后面计算 cosine similarity
class_centers = F.normalize(class_centers, dim=1)

print("训练集特征中心计算完成！")
print("class_centers shape:", class_centers.shape)

正在提取训练集特征，并计算每一类的特征中心...


Extract train features:   0%|          | 0/514 [00:00<?, ?it/s]

训练集特征中心计算完成！
class_centers shape: torch.Size([101, 2048])


## 5. Predict on test set

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
from tqdm.auto import tqdm
import pandas as pd

# ============================================================
# 配置
# ============================================================
NUM_CLASSES = NUM_KNOWN  # 101 已知类别
OOD_LABEL = 101
OOD_RATIO = 0.25  # 测试集中最低 25% 判 unknown

# class_centers: 训练集每类特征中心，shape [101, feature_dim]
# 已在提取训练特征代码块中计算完成

# ============================================================
# 测试阶段预测函数（特征距离 + 25%最低判 unknown）
# ============================================================

model.eval()
all_ids = []
all_preds = []

with torch.no_grad():
    for x, ids in tqdm(test_loader, desc="Predict test features"):
        x = x.to(DEVICE, non_blocking=True)

        # --- 方法一提取 ResNet fc 前特征 ---
        feats = extract_resnet_features(model, x)  # [batch_size, 2048]
        feats = F.normalize(feats, dim=1)         # 归一化

        # --- 计算与训练集每类中心的 cosine similarity ---
        sims = feats @ class_centers.T            # [batch_size, NUM_CLASSES]
        max_sim, pred = sims.max(dim=1)

        # --- 最低 25% 相似度判为 unknown ---
        threshold = np.quantile(max_sim.cpu().numpy(), OOD_RATIO)
        pred = torch.where(max_sim <= threshold, torch.full_like(pred, OOD_LABEL), pred)

        all_ids.extend(ids)
        all_preds.extend(pred.cpu().numpy().tolist())

Predict test features:   0%|          | 0/158 [00:00<?, ?it/s]

## 6. Submission

In [ ]:
sub = pd.DataFrame({'image_id': all_ids, 'label': all_preds})
# DataLoader 是 shuffle=False，但 num_workers>0 时返回顺序不保证完全一致
# 这里按 sample_submission.csv 的 image_id 顺序重排，确保提交对齐
order = pd.read_csv(DATA_ROOT / 'test.csv')['image_id'].tolist()
sub = sub.set_index('image_id').loc[order].reset_index()
sub.to_csv(OUT_SUB, index=False)
print(f'Wrote {OUT_SUB} ({len(sub)} rows)')
sub.head()

Wrote /content/Food-Classification/submission(6).csv (20179 rows)


,image_id,label
0,test_00000,89
1,test_00001,13
2,test_00002,49
3,test_00003,68
4,test_00004,52
